# 🎓 Morfología y Análisis de Contornos

En este tema aprenderemos a analizar la **forma** de los objetos. Una vez que hemos detectado bordes o colores, necesitamos entender la geometría: ¿Es un círculo? ¿Qué área tiene? ¿Dónde está su centro?

### 🎯 Objetivos de Aprendizaje
1.  Aplicar **Erosión y Dilatación** para limpiar máscaras.
2.  Entender **Apertura y Cierre** (Opening/Closing).
3.  Encontrar y dibujar **Contornos**.
4.  Calcular propiedades: Área, Perímetro y Centroide.

---

## 1. Operaciones Morfológicas

Son transformaciones basadas en la forma de la imagen. Se usan principalmente en imágenes binarias (máscaras) para eliminar ruido o separar objetos conectados.

* **Erosión:** "Come" los bordes de los objetos blancos. Útil para quitar ruido (puntos blancos pequeños).
* **Dilatación:** "Expande" los objetos blancos. Útil para rellenar huecos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Crear una imagen binaria con ruido (letra 'J' con puntos)
imagen = np.zeros((300, 300), dtype=np.uint8)
cv2.putText(imagen, 'J', (80, 200), cv2.FONT_HERSHEY_SIMPLEX, 7, 255, 15)

# Añadir ruido (puntos blancos aleatorios)
for i in range(200):
    y, x = np.random.randint(0, 300, 2)
    imagen[y, x] = 255

# Definir el Kernel (Elemento estructurante)
kernel = np.ones((5,5), np.uint8)

# 1. Erosión (Elimina el ruido pequeño, pero adelgaza la letra)
erosion = cv2.erode(imagen, kernel, iterations=1)

# 2. Dilatación (Engrosa la letra)
dilatacion = cv2.dilate(imagen, kernel, iterations=1)

# 3. Apertura (Erosión seguida de Dilatación)
# Es lo IDEAL para quitar ruido sin cambiar el tamaño final del objeto
apertura = cv2.morphologyEx(imagen, cv2.MORPH_OPEN, kernel)

plt.figure(figsize=(12, 4))
plt.subplot(1,4,1); plt.imshow(imagen, cmap='gray'); plt.title("Original con Ruido")
plt.subplot(1,4,2); plt.imshow(erosion, cmap='gray'); plt.title("Erosión")
plt.subplot(1,4,3); plt.imshow(dilatacion, cmap='gray'); plt.title("Dilatación")
plt.subplot(1,4,4); plt.imshow(apertura, cmap='gray'); plt.title("Apertura (Mejor)")
plt.show()

## 2. Contornos

Un contorno es una curva que une todos los puntos continuos (a lo largo del borde), que tienen el mismo color o intensidad. Es la herramienta fundamental para el análisis de formas.

In [ ]:
# Usaremos la imagen 'apertura' limpia del paso anterior

# Encontrar contornos
# RETR_EXTERNAL: Solo contornos externos (ignora huecos dentro de la letra)
# CHAIN_APPROX_SIMPLE: Comprime segmentos (ahorra memoria)
contornos, jerarquia = cv2.findContours(apertura, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Contornos encontrados: {len(contornos)}")

# Dibujar contornos
# Necesitamos convertir a color para ver el contorno dibujado en verde
img_color = cv2.cvtColor(apertura, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_color, contornos, -1, (0, 255, 0), 3) # -1 dibuja todos

plt.imshow(img_color)
plt.title("Contornos Detectados")
plt.show()

### 2.1 Propiedades de los Contornos
Podemos calcular matemáticas sobre estos contornos.

In [ ]:
cnt = contornos[0]

# 1. Área
area = cv2.contourArea(cnt)

# 2. Perímetro
perimetro = cv2.arcLength(cnt, True) # True = contorno cerrado

# 3. Bounding Box (Rectángulo que encierra el objeto)
x, y, w, h = cv2.boundingRect(cnt)

# 4. Centroide (Momentos)
M = cv2.moments(cnt)
if M['m00'] != 0:
    cx = int(M['m10']/M['m00'])
    cy = int(M['m01']/M['m00'])
else:
    cx, cy = 0, 0

print(f"Área: {area}")
print(f"Perímetro: {perimetro:.2f}")
print(f"Centro: ({cx}, {cy})")

# Visualizar Bounding Box y Centro
cv2.rectangle(img_color, (x, y), (x+w, y+h), (0, 0, 255), 2)
cv2.circle(img_color, (cx, cy), 5, (255, 0, 0), -1)

plt.imshow(img_color)
plt.title("Análisis de Forma")
plt.show()

## 3. 🚀 Mini-Proyecto: Analizador de Formas

**Objetivo:** Crea un script que analice una imagen con varias figuras geométricas (círculos, cuadrados, triángulos) y las clasifique y etiquete automáticamente.

**Lógica sugerida:**
1.  Detectar contornos.
2.  Usar `cv2.approxPolyDP` para simplificar la forma y contar vértices.
    * 3 vértices = Triángulo
    * 4 vértices = Cuadrado/Rectángulo
    * >8 vértices = Círculo

In [ ]:
# TODO: Implementa el mini-proyecto aquí

import cv2
import numpy as np

# --- 1. Generación de Imagen de Prueba (Formas Geométricas) ---
# Fondo negro
imagen = np.zeros((400, 400, 3), dtype=np.uint8)

# Dibujar Triángulo (Verde)
pts_triangulo = np.array([[100, 50], [50, 150], [150, 150]], np.int32)
cv2.fillPoly(imagen, [pts_triangulo], (0, 255, 0))

# Dibujar Cuadrado (Azul)
cv2.rectangle(imagen, (220, 50), (320, 150), (255, 0, 0), -1)

# Dibujar Rectángulo (Amarillo)
cv2.rectangle(imagen, (50, 250), (180, 320), (0, 255, 255), -1)

# Dibujar Círculo (Rojo)
cv2.circle(imagen, (300, 300), 50, (0, 0, 255), -1)

print("📐 Analizando geometría de la imagen...")

# --- 2. Procesamiento y Detección ---

# Convertir a escala de grises
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Binarizar (Threshold)
# Como el fondo es negro (0) y las figuras tienen color, un umbral bajo (20) sirve
_, thresh = cv2.threshold(gris, 20, 255, cv2.THRESH_BINARY)

# Encontrar contornos
contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"   > Se encontraron {len(contornos)} objetos.")

# --- 3. Clasificación de Formas ---

for cnt in contornos:
    # Calcular perímetro
    perimetro = cv2.arcLength(cnt, True)
    
    # Aproximación Poligonal (Clave del ejercicio)
    # epsilon es la precisión: mientras más pequeño, más se ajusta a la curva original.
    # 0.04 (4%) es un buen estándar para distinguir formas geométricas simples.
    epsilon = 0.04 * perimetro
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    
    # El número de vértices nos dice qué forma es
    vertices = len(approx)
    
    # Obtener el centro para poner el texto (Momentos)
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10']/M['m00'])
        cy = int(M['m01']/M['m00'])
    else:
        continue
        
    forma_nombre = "Desconocido"
    
    # Lógica de clasificación
    if vertices == 3:
        forma_nombre = "Triangulo"
        
    elif vertices == 4:
        # Diferenciar Cuadrado de Rectángulo usando Aspect Ratio (Proporción Ancho/Alto)
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h
        
        # Si el ancho es casi igual al alto (con 5% de margen), es cuadrado
        if 0.95 <= aspect_ratio <= 1.05:
            forma_nombre = "Cuadrado"
        else:
            forma_nombre = "Rectangulo"
            
    elif vertices > 4:
        # En geometría digital, un círculo es un polígono con muchos lados
        forma_nombre = "Circulo"
        
    print(f"   > Objeto en ({cx},{cy}): {forma_nombre} (Vértices detectados: {vertices})")
    
    # --- 4. Visualización ---
    
    # Dibujar el contorno aproximado (blanco, grosor 3)
    cv2.drawContours(imagen, [approx], 0, (255, 255, 255), 3)
    
    # Poner etiqueta de texto
    cv2.putText(imagen, forma_nombre, (cx-40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

# Mostrar resultado final
cv2.imshow("Clasificacion de Formas", imagen)

print("\nPresiona cualquier tecla para cerrar...")
cv2.waitKey(0)
cv2.destroyAllWindows()

📐 Analizando geometría de la imagen...
   > Se encontraron 4 objetos.
   > Objeto en (300,300): Circulo (Vértices detectados: 8)
   > Objeto en (115,285): Rectangulo (Vértices detectados: 4)
   > Objeto en (270,100): Cuadrado (Vértices detectados: 4)
   > Objeto en (99,116): Triangulo (Vértices detectados: 3)

Presiona cualquier tecla para cerrar...
